<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_Mini8projet_W8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BaristaBot: Building a Stateful Agent with LangGraph and Gemini

This notebook guides you through building a conversational cafe ordering system. We will use **LangGraph** for state management and **Gemini** for the intelligence.

### 1. Install Dependencies

In [ ]:
%pip install -qU "langgraph==0.2.60" "langchain-google-genai==2.0.8" "google-genai==1.1.0"

### 2. Setup API Key
We'll use Colab's secret manager to securely handle the API key.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except:
    print("Please add your GOOGLE_API_KEY to Colab Secrets.")

### 3. Define State and Instructions
LangGraph uses a `TypedDict` to track the state of the conversation (messages, current order, and completion status).

In [ ]:
from typing import Annotated, Literal, Iterable
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class OrderState(TypedDict):
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool

BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. "
    "Use add_to_order to add items and place_order to finish. "
    "Always confirm the order before placing it."
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

### 4. Define Tools and Nodes
We define the tools the bot can use (like getting the menu) and the nodes (steps) in our graph.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolMessage
from random import randint

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return "Coffee: Espresso, Latte, Americano. Tea: Green Tea, Chai Latte. Modifiers: Oat milk, Vanilla."

@tool
def add_to_order(drink: str, modifiers: list[str]) -> str:
    """Adds drink to order."""
    return f"Added {drink}"

@tool
def place_order() -> int:
    """Finalizes the order."""
    return randint(1, 10)

# Group tools
auto_tools = [get_menu]
order_tools = [add_to_order, place_order]
tools = auto_tools + order_tools
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: OrderState):
    if not state["messages"]:
        return {"messages": [AIMessage(content=WELCOME_MSG)], "order": [], "finished": False}
    response = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    return {"messages": [response]}

### 5. Build the Graph
Now we connect the nodes into a workflow.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(OrderState)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")

def route(state):
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return END

builder.add_conditional_edges("chatbot", route)
builder.add_edge("tools", "chatbot")

app = builder.compile()

### 6. Run the Bot
You can now interact with the compiled graph.

In [ ]:
final_state = app.invoke({"messages": [("user", "What is on the menu?")]})
for msg in final_state["messages"]:
    print(f"{msg.type}: {msg.content}")